# Phase 3 baseline: Colab pipeline

Runs top to bottom, no cell edits, on a **T4 GPU runtime** (Runtime > Change runtime type).

This machine has no GPU and a slow connection (~1.25 MB/s to the HF CDN, `results/throughput_laptop.json`), so everything below runs here instead. Git is the bridge: this notebook clones the **public** repo anonymously (no token needed), and at the end copies results back to Drive for a laptop to commit -- Colab never pushes (see HANDOFF.md "Workflow").

One deviation from the plan as originally written: PLAN.md sketches "download" and "normalize" as two separate steps. In this codebase they are not separable -- `scripts/download_data.py` calls the canonical decode path (`src/data/normalize.py`) inline on every row as it streams the parquet, and never writes a raw file to disk (Phase 2 design: "nothing raw is ever written"). So Cell 2 below does both at once; there is no standalone normalization pass to run afterward.

## Cell 1 -- mount Drive, clone, install, verify the bridge

Runs `scripts/bench_throughput.py` last, so a broken clone/install/GPU fails here in well under a minute rather than 10+ minutes into a download.

In [19]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/anniketkumar/aigc-detect.git"

REPO_DIR = "/content/repo"
DRIVE_ROOT = "/content/drive/MyDrive/aigc"

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/features", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/results", exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
# Public repo -- anonymous clone, no auth, no token.
!git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!pip install -q -r requirements.txt

fatal: destination path '/content/repo' already exists and is not an empty directory.
/content/repo


In [21]:
import torch

has_gpu = torch.cuda.is_available()
print("CUDA available:", has_gpu)
print("GPU:", torch.cuda.get_device_name(0) if has_gpu else "NONE")
assert has_gpu, (
    "No GPU visible. Runtime > Change runtime type > T4 GPU, then Runtime > "
    "Restart session and rerun from the top. A CPU runtime will make cell 4 "
    "(CLIP feature caching) take hours instead of minutes."
)

CUDA available: True
GPU: Tesla T4


In [22]:
# The bridge check. If this fails or reports laptop-like throughput, stop and
# fix the environment before spending time on cell 2's download.
!python -m scripts.bench_throughput --out results/throughput_colab.json

{
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "python": "3.13.15",
  "cpu_count": 2,
  "in_colab": true,
  "in_kaggle": true,
  "torch": "2.13.0+cu130",
  "cuda": true,
  "gpu": "Tesla T4",
  "gpu_mem_gb": 15.6
}

[cdn] SID_Set ...
            28.31 MB/s  (102 GB/h)   {'file_bytes': 477663216, 'chunk_mb': 50.3, 'reps': 3, 'mbps_median': 28.31, 'mbps_all': [25.42, 28.31, 28.38]}
[parquet] SID_Set ...
                         0.00 MB/s   {'error': "HfHubHTTPError: (Request ID: Root=1-6a9d7bc8-13fac3ee73fefcf171b1b19f;3149e4e6-1375-45ae-a66e-8ebfcaa2d26b)\n\n429 Too Many Requests: you have reached your 'api' rate limit.\nRetry after 128 seconds (499/500 requests remaining in current 300s window).\nUrl: https://huggingface.co/api/datasets/saberzl/SID_Set.\nmaximum queue size reached"}

[cdn] GenImage_BigGAN ...
            29.71 MB/s  (107 GB/h)   {'file_bytes': 652468032, 'chunk_mb': 50.3, 'reps': 3, 'mbps_median': 29.71, 'mbps_all': [32.52, 27.72, 29.71]}
[parquet] GenImage_B

## Cell 2 -- download + normalize (one step, see the note above), build manifests

Uses the default per-source quotas in `src/data/sources.py`: ~18.2k images, 4 training generators + 3 held-out generators (MidJourney, Gemini, FLUX.1-dev) + 3 real sources. Images land in `/content` (ephemeral) -- nothing here goes to Drive, per the Workflow note (free tier is 15 GB and images get re-pulled every session anyway).

In [23]:
!python -m scripts.download_data --out data/corpus

source                   lab  quota  MB/img transfer GB  disk GB
SDXL                       1   2000   1.392        3.28    0.190
Mobius                     1   2000   1.527        3.59    0.190
RealVisXL-V4.0             1   2000   1.422        3.35    0.190
Aura                       1   2000   0.505        1.19    0.190
MidJourney                 1   2000   0.751        1.77    0.190
Gemini-nano-banana         1   2000   1.576        3.71    0.190
FLUX.1-dev                 1    800   0.566        0.53    0.076
OpenImagesV7               0   1800   0.566        1.20    0.171
Megalith-Flickr            0   1800   0.194        0.41    0.171
Unsplash                   0   1800   0.130        0.28    0.171

  images      18,200
  transfer    19.31 GB  (network)
  on disk     1.73 GB  (normalized, kept)
  peak disk   1.93 GB

config 62e4f79471ec; ledger has 18200 rows

[1/10] OpenImagesV7 (saberzl/SID_Set default/validation)
  OpenImagesV7: 1800/1800 already done
[2/10] Megalith-Flickr (

In [24]:
!python -m src.data.manifest --ledger data/corpus/ledger.csv --out data/manifests
!python -m scripts.make_data_stats

  hashed 2000/18200
  hashed 4000/18200
  hashed 6000/18200
  hashed 8000/18200
  hashed 10000/18200
  hashed 12000/18200
  hashed 14000/18200
  hashed 16000/18200
  hashed 18000/18200
{
  "n_images": 18200,
  "splits": {
    "train": 9380,
    "val": 2010,
    "test": 6810
  },
  "n_clusters": 18198,
  "n_dup_pairs": 2,
  "n_blocked": 2,
  "blocklist_empty": false,
  "blocklist_sources": [
    "COCO val2017: 5000 images, 5000 sha256, 5000 phash"
  ],
  "blocklist_gaps": [
    "DALL-E Advanced (8,843 imgs): no public standalone distribution -- it ships only inside WildFake's ModelScope zips (~700 GB), which is not a feasible download here. Not hashed. Mitigated by the source-registry denylist asserted in tests/test_manifest.py::test_no_dalle_derived_source_in_the_registry."
  ],
  "holdout_generators": [
    "FLUX.1-dev",
    "Gemini-nano-banana",
    "MidJourney"
  ],
  "by_split_label": {
    "0": {
      "test": 810,
      "train": 3780,
      "val": 810
    },
    "1": {
      "tes

## Gate -- the two non-negotiable guards, plus the rest of the suite

`tests/test_manifest.py::test_real_corpus_manifest_builds_and_holds_every_invariant` only *engages* once `data/corpus/ledger.csv` exists (HANDOFF.md's documented known gap, `tests/test_manifest.py:456`) -- which is exactly now, for the first time this session. This is the one place in the whole pipeline where the real WildFake/DALL·E blocklist is actually exercised against real data. **Do not skip this cell or continue past a failure.**

In [25]:
import subprocess

# Colab does not capture a child process's fd 1, so the previous run's gate
# recorded nothing but its own success line -- exit 0 proved the print ran, not
# that the guards engaged. Capture the output, show the tail, and pass -rs so
# skips are visible: both guards are skipif-gated on data/corpus/ledger.csv.
result = subprocess.run(
    ["python", "-m", "pytest", "-q", "-rs"], capture_output=True, text=True
)
print(result.stdout[-4000:])
if result.stderr.strip():
    print("--- stderr (tail) ---")
    print(result.stderr[-2000:])
if result.returncode != 0:
    raise SystemExit(
        "pytest failed. Do not continue past this cell -- this is the run where "
        "test_manifest.py's WildFake content-hash guard and the DALL·E "
        "source-registry denylist test actually see real data (HANDOFF.md "
        "'Non-negotiable'). A failure here is a disqualification risk, not a "
        "flaky test to retry past."
    )

# Exit 0 alone is not evidence: a skipif-skipped guard also exits 0. Name the
# two non-negotiables and require PASSED, so the claim below is witnessed.
GUARDS = [
    "tests/test_manifest.py::test_real_corpus_manifest_builds_and_holds_every_invariant",
    "tests/test_manifest.py::test_no_dalle_derived_source_in_the_registry",
]
witness = subprocess.run(
    ["python", "-m", "pytest", "-v", "-rs", *GUARDS], capture_output=True, text=True
)
print(witness.stdout[-2500:])
n_passed = witness.stdout.count("PASSED")
if witness.returncode != 0 or n_passed != len(GUARDS):
    raise SystemExit(
        f"the WildFake/DALL·E guards did not both PASS ({n_passed}/{len(GUARDS)}). "
        "If they SKIPPED, data/corpus/ledger.csv is missing and the "
        "non-negotiable check never ran -- HANDOFF.md, disqualification risk."
    )
print(
    f"pytest green, and both real-corpus guards PASSED "
    f"({n_passed}/{len(GUARDS)}) -- witnessed by name, not assumed"
)

........................................................................ [ 20%]
........................................................................ [ 40%]
........................................................................ [ 60%]
........................................................................ [ 81%]
...................................................................      [100%]
355 passed in 510.04s (0:08:30)

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-9.1.1, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/repo
plugins: anyio-4.14.2, langsmith-0.11.1, typeguard-4.6.0
collecting ... collected 2 items

tests/test_manifest.py::test_real_corpus_manifest_builds_and_holds_every_invariant PASSED [ 50%]
tests/test_manifest.py::test_no_dalle_derived_source_in_the_registry PASSED [100%]

======================== 2 passed in 122.99s (0:02:02) =========================

pyte

## Cell 4 -- cache frozen CLIP ViT-B/16 features to Drive

One pass per split. After this, every later head experiment (Phase 3 baseline, and any Phase 4+ head swap) reads three small `.npy`/`.json` files and needs no GPU.

In [26]:
FEATURES_DIR = f"{DRIVE_ROOT}/features"

for split in ("train", "val", "test"):
    !python -m scripts.cache_features \
        --manifest data/manifests/{split}.csv \
        --out {FEATURES_DIR}/{split} \
        --device cuda --batch-size 256

loading ViT-B-16-quickgelu (openai) on cuda ...
  backbone ready in 9.1s, embed_dim=512

{
  "manifest": "data/manifests/train.csv",
  "backbone": "ViT-B-16-quickgelu",
  "pretrained": "openai",
  "clip_embed_dim": 512,
  "embed_dim": 512,
  "n_images": 9380,
  "n_failed": 0,
  "n_real": 3780,
  "n_ai": 5600,
  "augment_copies": 0,
  "augment_seed": null,
  "fuse_freq": false,
  "freq_dim": 0,
  "elapsed_s": 189.6
}
-> /content/drive/MyDrive/aigc/features/train
loading ViT-B-16-quickgelu (openai) on cuda ...
  backbone ready in 9.3s, embed_dim=512

{
  "manifest": "data/manifests/val.csv",
  "backbone": "ViT-B-16-quickgelu",
  "pretrained": "openai",
  "clip_embed_dim": 512,
  "embed_dim": 512,
  "n_images": 2010,
  "n_failed": 0,
  "n_real": 810,
  "n_ai": 1200,
  "augment_copies": 0,
  "augment_seed": null,
  "fuse_freq": false,
  "freq_dim": 0,
  "elapsed_s": 50.1
}
-> /content/drive/MyDrive/aigc/features/val
loading ViT-B-16-quickgelu (openai) on cuda ...
  backbone ready in 7.9s, 

## Cell 5 -- train the baseline head

Linear head, BCE, no augmentation -- the deliberate control (HANDOFF.md), now trained to convergence (300 epochs, early stopping patience 30 on val AUROC). Checkpoint goes straight to Drive, not to `/content`, because free Colab disconnects on idle and this is cheap to lose only once.

In [27]:
CKPT = f"{DRIVE_ROOT}/checkpoints/baseline.pt"

# 300 epochs with early stopping (patience 30 on val AUROC), not a flat 50: the
# first run's val AUROC was still climbing at epoch 49, so best-val selection
# returned the final epoch and never selected anything. An unconverged control
# inflates Phase 4's apparent augmentation gain. Costs ~4 seconds.
!python -m src.train \
    --features-dir {FEATURES_DIR} \
    --train-split train --val-split val \
    --out {CKPT} \
    --epochs 300 --patience 30 --device cuda

epoch   0  loss 0.6729  train_auroc 0.9292  val_auroc 0.9313
epoch   1  loss 0.6405  train_auroc 0.9495  val_auroc 0.9518
epoch   2  loss 0.6150  train_auroc 0.9568  val_auroc 0.9593
epoch   3  loss 0.5932  train_auroc 0.9601  val_auroc 0.9625
epoch   4  loss 0.5736  train_auroc 0.9623  val_auroc 0.9645
epoch   5  loss 0.5556  train_auroc 0.9637  val_auroc 0.9661
epoch   6  loss 0.5386  train_auroc 0.9646  val_auroc 0.9670
epoch   7  loss 0.5229  train_auroc 0.9653  val_auroc 0.9678
epoch   8  loss 0.5081  train_auroc 0.9660  val_auroc 0.9685
epoch   9  loss 0.4942  train_auroc 0.9666  val_auroc 0.9691
epoch  10  loss 0.4811  train_auroc 0.9671  val_auroc 0.9696
epoch  11  loss 0.4689  train_auroc 0.9676  val_auroc 0.9701
epoch  12  loss 0.4574  train_auroc 0.9680  val_auroc 0.9705
epoch  13  loss 0.4465  train_auroc 0.9685  val_auroc 0.9710
epoch  14  loss 0.4362  train_auroc 0.9689  val_auroc 0.9713
epoch  15  loss 0.4266  train_auroc 0.9693  val_auroc 0.9718
epoch  16  loss 0.4174  

## Cell 6 -- run the Phase 1 harness on the held-out test split

`--split test` includes both the transform grid *and* the three held-out generators (MidJourney, Gemini, FLUX.1-dev), since `src/data/manifest.py` routes them there in full.

In [28]:
!python -m src.evaluate \
    --model clip_linear --ckpt {CKPT} \
    --split test --out results/baseline/ \
    --device cuda

model=clip_linear(/content/drive/MyDrive/aigc/checkpoints/baseline.pt)  images=6810 (real=810, ai=6000)  cells=19  seed=0
AUROC null SD at this sample size: 0.0108 (a chance-level model should land within ~0.032 of 0.500)


clean AUROC        0.9810
mean transformed   0.9710   (family-balanced over 7 families)
robustness_gap     +0.0099   (lower is better)
worst_case         0.9484   (composed_resize0.25+blur0.5+jpeg30)
  flat mean        0.9689   gap +0.0120   (§3.2 literal, secondary)

wrote results/baseline/grid.csv and results/baseline/report.md in 3372.07s


In [29]:
import json

summary = json.loads(open("results/baseline/summary.json").read())["summary"]
clean_auroc = summary["clean_auroc"]
print(f"clean AUROC: {clean_auroc:.4f}")

if clean_auroc > 0.99:
    print(
        "\n*** STOP AND FLAG THIS ***\n"
        "HANDOFF.md: clean AUROC above 0.99 is far more likely to be a "
        "surviving leak than a good model, given what the Phase 2 audit found "
        "in raw SID_Set. Do not treat this as a good result -- check the "
        "held-out generator breakdown and re-run the Phase 2 normalization "
        "audit (tests/test_normalization_audit.py) before trusting it."
    )
elif not (0.85 <= clean_auroc <= 0.95):
    print(
        f"note: clean AUROC {clean_auroc:.4f} is outside the expected "
        "0.85-0.95 band (HANDOFF.md). Not necessarily wrong -- just worth a "
        "second look before reporting it as the baseline."
    )

clean AUROC: 0.9810
note: clean AUROC 0.9810 is outside the expected 0.85-0.95 band (HANDOFF.md). Not necessarily wrong -- just worth a second look before reporting it as the baseline.


## Cell 6b -- aesthetic probe: how much of the number is "looks artistic"?

Two pixel features (`hf_energy`, `flat_frac`), logistic regression, no CLIP. The first run had unseen MidJourney (0.9894) beating every trained-on generator while unseen FLUX sat at 0.433 TPR@1% -- the signature of a style detector, not a forensic one. If two texture statistics reproduce that ordering, the headline is substantially aesthetics.

Runs after cell 6 on purpose: it reads `results/baseline/scores.csv` to put the CLIP baseline's per-generator AUROC beside its own. CPU only, ~3-5 min.

In [30]:
# --verify-against-audit first checks these two features match
# scripts.audit_leakage.content_features exactly, so the probe is the Phase 2
# measure and not a second invented one.
!python -m scripts.aesthetic_probe \
    --manifests data/manifests \
    --out results/baseline \
    --verify-against-audit 20


verified 20 image(s): both feature paths agree exactly
measuring 9380 train + 6810 test images on hf_energy, flat_frac ...


{
  "overall": {
    "auroc": 0.6085399176954733,
    "tpr_at_fpr1": 0.03216666666666667
  },
  "spearman_vs_clip": {
    "rho": -0.14285714285714288,
    "p": 0.7599453002180929,
    "n_generators": 7
  },
  "elapsed_s": 150.3
}
-> results/baseline/aesthetic_probe.md


## Cell 7 -- copy results back to Drive (no git push from Colab)

A laptop pulls this folder from Drive and commits it -- Colab never pushes, which removes an auth failure mode deliberately (HANDOFF.md "Workflow").

In [31]:
import shutil

shutil.copytree("results/baseline", f"{DRIVE_ROOT}/results/baseline", dirs_exist_ok=True)
shutil.copy("results/data_stats.md", f"{DRIVE_ROOT}/results/data_stats.md")
print(f"copied to {DRIVE_ROOT}/results/baseline and {DRIVE_ROOT}/results/data_stats.md")
print("On the laptop: pull these from Drive into results/baseline/ and results/data_stats.md, then git add + commit + push.")

copied to /content/drive/MyDrive/aigc/results/baseline and /content/drive/MyDrive/aigc/results/data_stats.md
On the laptop: pull these from Drive into results/baseline/ and results/data_stats.md, then git add + commit + push.


## Phase 4 -- augmentation: cache K augmented copies, retrain, evaluate into results/aug/

Continues in the *same* session as Phase 3 above -- it reuses `data/corpus` (still on `/content` from cell 2) and `data/manifests` (cell 3), so this block only works run top-to-bottom after Phase 3, not standalone in a fresh session.

`src/data/augment.py` (PLAN.md §6): continuous severities *wider* than the eval grid -- jpeg q ∈ [20,95] vs the eval grid's {90,70,50,30}, and similarly for blur/resize/noise/jitter/crop -- 1-3 families per image, random order, ~20% left clean, K=4 copies/image. Only `train` is augmented: `val` reuses the clean Phase 3 cache as-is, because early stopping needs to select on the real distribution, not the one being trained on, and `test` is untouched raw images, so `results/aug/` is a direct apples-to-apples comparison against `results/baseline/`.

In [32]:
FEATURES_AUG_DIR = f"{DRIVE_ROOT}/features_aug"
AUG_COPIES = 4

# --batch-size 64: with --augment-copies 4 that's 256 images per clip.embed() call,
# the same per-call size as Phase 3's plain --batch-size 256.
!python -m scripts.cache_features \n    --manifest data/manifests/train.csv \n    --out {FEATURES_AUG_DIR}/train \n    --device cuda --batch-size 64 \n    --augment-copies {AUG_COPIES} --augment-seed 0

# val stays clean -- copy the Phase 3 cache rather than re-embedding it.
import shutil
shutil.copytree(f"{FEATURES_DIR}/val", f"{FEATURES_AUG_DIR}/val", dirs_exist_ok=True)

usage: cache_features.py [-h] --manifest MANIFEST --out OUT [--device DEVICE]
                         [--backbone BACKBONE] [--pretrained PRETRAINED]
                         [--batch-size BATCH_SIZE] [--limit LIMIT] [--quiet]
                         [--augment-copies AUGMENT_COPIES]
                         [--augment-seed AUGMENT_SEED] [--fuse-freq]
cache_features.py: error: unrecognized arguments: n n n n


'/content/drive/MyDrive/aigc/features_aug/val'

### Retrain the head on augmented features

Same linear head, same training script, same convergence policy (300 epochs, patience 30) -- only the features underneath changed. Checkpoint goes to Drive under its own name so `baseline.pt` is never overwritten.

In [33]:
CKPT_AUG = f"{DRIVE_ROOT}/checkpoints/aug.pt"

!python -m src.train \n    --features-dir {FEATURES_AUG_DIR} \n    --train-split train --val-split val \n    --out {CKPT_AUG} \n    --epochs 300 --patience 30 --device cuda

usage: train.py [-h] [--features-dir FEATURES_DIR] [--train-split TRAIN_SPLIT]
                [--val-split VAL_SPLIT] [--out OUT] [--epochs EPOCHS]
                [--patience PATIENCE] [--lr LR] [--weight-decay WEIGHT_DECAY]
                [--batch-size BATCH_SIZE] [--device DEVICE] [--seed SEED]
train.py: error: unrecognized arguments: n n n n


### Run the Phase 1 harness on the same held-out test split

`--split test` is untouched: same manifest, same three held-out generators (MidJourney, Gemini, FLUX.1-dev), same 19-cell grid as `results/baseline/`.

In [34]:
!python -m src.evaluate \n    --model clip_linear --ckpt {CKPT_AUG} \n    --split test --out results/aug/ \n    --device cuda

usage: python -m src.evaluate [-h]
                              [--model {clip_freq_fusion,clip_linear,dummy_brightness,dummy_random}]
                              [--ckpt CKPT] [--split SPLIT]
                              [--manifest MANIFEST]
                              [--manifest-root MANIFEST_ROOT] --out OUT
                              [--name NAME] [--cells [CELL_OR_FAMILY ...]]
                              [--limit LIMIT] [--batch-size BATCH_SIZE]
                              [--device DEVICE] [--seed SEED]
                              [--fpr-target FPR_TARGET]
                              [--cache-dir CACHE_DIR] [--no-scores] [--quiet]
python -m src.evaluate: error: unrecognized arguments: n n n


In [35]:
import json

aug_summary = json.loads(open("results/aug/summary.json").read())["summary"]
base_summary = json.loads(open("results/baseline/summary.json").read())["summary"]

print(f"clean AUROC           baseline {base_summary['clean_auroc']:.4f}  ->  aug {aug_summary['clean_auroc']:.4f}")
print(f"AUROC robustness_gap  baseline {base_summary['robustness_gap']:.4f}  ->  aug {aug_summary['robustness_gap']:.4f}")
print(f"worst_case AUROC      baseline {base_summary['worst_case']:.4f}  ->  aug {aug_summary['worst_case']:.4f}  ({aug_summary['worst_cell']})")

if aug_summary["clean_auroc"] > 0.99:
    print("*** STOP AND FLAG THIS ***","HANDOFF.md: clean AUROC above 0.99 is far more likely to be a surviving ","leak than a good model. Do not treat this as a good result.")

clean AUROC           baseline 0.9810  ->  aug 0.9779
AUROC robustness_gap  baseline 0.0099  ->  aug 0.0077
worst_case AUROC      baseline 0.9484  ->  aug 0.9526  (noise_0.1)


### TPR@FPR robustness gap -- the headline metric, not AUROC

Phase 3's analysis (`results/tpr_analysis/report.md`) found the AUROC gap (0.0099) sits under its own null SD (0.0108) -- statistically indistinguishable from zero -- while TPR@FPR=5% resolves cleanly: gap **0.0530**, 95% CI [0.0369, 0.0632], clearly nonzero. That is the metric augmentation is judged against, not the AUROC gap printed above. This reuses `scripts/tpr_gap_analysis.py` unchanged, just pointed at both runs' `scores.csv` so the two land in one comparison report.

In [36]:
!python -m scripts.tpr_gap_analysis \n    --run "aug" results/aug/scores.csv \n    --run "baseline" results/baseline/scores.csv \n    --out results/tpr_analysis_aug

usage: tpr_gap_analysis.py [-h] [--run LABEL SCORES_CSV]
                           [--pair LABEL_A SCORES_CSV_A LABEL_B SCORES_CSV_B]
                           --out OUT [--b-main B_MAIN] [--b-gen B_GEN]
                           [--b-pair B_PAIR] [--seed SEED]
tpr_gap_analysis.py: error: unrecognized arguments: n n n


### Per-generator TPR spread -- the other axis AUROC hides

`results/baseline/per_generator.md`: per-generator clean AUROC sits in a tight 0.9697-0.9949 band, but clean TPR@1% spans **0.511** (FLUX.1-dev, held out) to **0.893** (Aura) -- a 15x wider spread that AUROC never shows. Augmentation is judged on this too: not just whether the family-averaged TPR@5% gap above shrinks, but whether it narrows this spread rather than, say, buying the average down by improving the easy generators while leaving FLUX.1-dev where it was.

In [38]:
import pandas as pd

aug_gen = pd.read_csv("results/tpr_analysis_aug/tpr_per_generator_aug.csv")
base_gen = pd.read_csv("results/tpr_analysis_aug/tpr_per_generator_baseline.csv")

def spread(df, col):
    return float(df[col].max() - df[col].min())

for col in ("clean_tpr1", "clean_tpr5"):
    b, a = spread(base_gen, col), spread(aug_gen, col)
    verdict = "narrower" if a < b else "WIDER -- augmentation made this worse"
    print(f"{col} per-generator spread   baseline {b:.4f}  ->  aug {a:.4f}  ({verdict})")

cols = ["generator", "held_out", "clean_tpr1", "clean_tpr5"]
print("baseline:", base_gen[cols].to_string(index=False))
print("aug:", aug_gen[cols].to_string(index=False))

clean_tpr1 per-generator spread   baseline 0.3821  ->  aug 0.3758  (narrower)
clean_tpr5 per-generator spread   baseline 0.1365  ->  aug 0.1445  (WIDER -- augmentation made this worse)
baseline:          generator  held_out  clean_tpr1  clean_tpr5
              Aura     False    0.893333    0.973333
        FLUX.1-dev      True    0.511250    0.837500
Gemini-nano-banana      True    0.672000    0.863500
        MidJourney      True    0.848000    0.974000
            Mobius     False    0.826667    0.950000
    RealVisXL-V4.0     False    0.796667    0.916667
              SDXL     False    0.783333    0.890000
aug:          generator  held_out  clean_tpr1  clean_tpr5
              Aura     False    0.913333    0.963333
        FLUX.1-dev      True    0.537500    0.837500
Gemini-nano-banana      True    0.672500    0.831500
        MidJourney      True    0.886500    0.976000
            Mobius     False    0.833333    0.943333
    RealVisXL-V4.0     False    0.803333    0.910000
     

### Copy Phase 4 results back to Drive

Same convention as Phase 3 -- Colab never pushes; a laptop pulls this from Drive and commits.

In [39]:
shutil.copytree("results/aug", f"{DRIVE_ROOT}/results/aug", dirs_exist_ok=True)
shutil.copytree("results/tpr_analysis_aug", f"{DRIVE_ROOT}/results/tpr_analysis_aug", dirs_exist_ok=True)
print(f"copied to {DRIVE_ROOT}/results/aug and {DRIVE_ROOT}/results/tpr_analysis_aug")
print("On the laptop: pull these from Drive into results/aug/ and results/tpr_analysis_aug/, then git add + commit + push.")

copied to /content/drive/MyDrive/aigc/results/aug and /content/drive/MyDrive/aigc/results/tpr_analysis_aug
On the laptop: pull these from Drive into results/aug/ and results/tpr_analysis_aug/, then git add + commit + push.


## Phase 5 -- frequency fusion: cache fused features, retrain, evaluate into results/fusion/

Continues in the *same* session as Phase 3/4 above -- reuses `data/manifests` from cell 3. Standalone-safe otherwise: unlike Phase 4 this does not depend on Phase 4 having run first.

`src/features/frequency.py`: real FFT radial-ring + block-DCT features, concatenated onto the CLIP embedding via `scripts/cache_features.py --fuse-freq`. `src/train.py` is unchanged -- it already reads `embed_dim` off the cached array's shape -- so this is the same training call as Phase 3/4, just pointed at a wider features directory. Score the result with `clip_freq_fusion`, not `clip_linear` (`src/models/clip_fusion.py` rejects a mismatched checkpoint loudly rather than silently misapplying it).

Validated end-to-end offline on a 303-image local smoke manifest before this cell was written (pipeline correctness only -- too small an N to say anything about whether fusion helps; that question is what this cell, at full 18.2k-image scale, actually answers).

In [40]:
FEATURES_FUSION_DIR = f"{DRIVE_ROOT}/features_fusion"

for split in ("train", "val", "test"):
    !python -m scripts.cache_features \
        --manifest data/manifests/{split}.csv \
        --out {FEATURES_FUSION_DIR}/{split} \
        --device cuda --batch-size 256 --fuse-freq

loading ViT-B-16-quickgelu (openai) on cuda ...
  backbone ready in 9.4s, embed_dim=512

{
  "manifest": "data/manifests/train.csv",
  "backbone": "ViT-B-16-quickgelu",
  "pretrained": "openai",
  "clip_embed_dim": 512,
  "embed_dim": 531,
  "n_images": 9380,
  "n_failed": 0,
  "n_real": 3780,
  "n_ai": 5600,
  "augment_copies": 0,
  "augment_seed": null,
  "fuse_freq": true,
  "freq_dim": 19,
  "elapsed_s": 250.4
}
-> /content/drive/MyDrive/aigc/features_fusion/train
loading ViT-B-16-quickgelu (openai) on cuda ...
  backbone ready in 9.6s, embed_dim=512

{
  "manifest": "data/manifests/val.csv",
  "backbone": "ViT-B-16-quickgelu",
  "pretrained": "openai",
  "clip_embed_dim": 512,
  "embed_dim": 531,
  "n_images": 2010,
  "n_failed": 0,
  "n_real": 810,
  "n_ai": 1200,
  "augment_copies": 0,
  "augment_seed": null,
  "fuse_freq": true,
  "freq_dim": 19,
  "elapsed_s": 61.1
}
-> /content/drive/MyDrive/aigc/features_fusion/val
loading ViT-B-16-quickgelu (openai) on cuda ...
  backbone r

### Retrain the head on fused features

Same linear head, same training script, same convergence policy (300 epochs, patience 30) -- only the features underneath are wider (512 CLIP dims + 19 frequency dims). Checkpoint goes to Drive under its own name so `baseline.pt`/`aug.pt` are never touched.

In [41]:
CKPT_FUSION = f"{DRIVE_ROOT}/checkpoints/fusion.pt"

!python -m src.train \
    --features-dir {FEATURES_FUSION_DIR} \
    --train-split train --val-split val \
    --out {CKPT_FUSION} \
    --epochs 300 --patience 30 --device cuda

epoch   0  loss 0.6636  train_auroc 0.8981  val_auroc 0.8881
epoch   1  loss 0.6271  train_auroc 0.8933  val_auroc 0.8827
epoch   2  loss 0.5977  train_auroc 0.8974  val_auroc 0.8870
epoch   3  loss 0.5725  train_auroc 0.9004  val_auroc 0.8902
epoch   4  loss 0.5506  train_auroc 0.9058  val_auroc 0.8960
epoch   5  loss 0.5316  train_auroc 0.9117  val_auroc 0.9023
epoch   6  loss 0.5142  train_auroc 0.9149  val_auroc 0.9058
epoch   7  loss 0.4984  train_auroc 0.9199  val_auroc 0.9115
epoch   8  loss 0.4841  train_auroc 0.9250  val_auroc 0.9170
epoch   9  loss 0.4709  train_auroc 0.9291  val_auroc 0.9216
epoch  10  loss 0.4586  train_auroc 0.9329  val_auroc 0.9261
epoch  11  loss 0.4473  train_auroc 0.9372  val_auroc 0.9313
epoch  12  loss 0.4365  train_auroc 0.9409  val_auroc 0.9355
epoch  13  loss 0.4263  train_auroc 0.9434  val_auroc 0.9386
epoch  14  loss 0.4169  train_auroc 0.9466  val_auroc 0.9427
epoch  15  loss 0.4079  train_auroc 0.9494  val_auroc 0.9462
epoch  16  loss 0.3994  

### Run the Phase 1 harness on the same held-out test split

`--split test` is untouched: same manifest, same three held-out generators (MidJourney, Gemini, FLUX.1-dev), same 19-cell grid as `results/baseline/` and `results/aug/`. `--model clip_freq_fusion`, not `clip_linear`.

In [42]:
!python -m src.evaluate \
    --model clip_freq_fusion --ckpt {CKPT_FUSION} \
    --split test --out results/fusion/ \
    --device cuda

model=clip_freq_fusion(/content/drive/MyDrive/aigc/checkpoints/fusion.pt)  images=6810 (real=810, ai=6000)  cells=19  seed=0
AUROC null SD at this sample size: 0.0108 (a chance-level model should land within ~0.032 of 0.500)


clean AUROC        0.9786
mean transformed   0.9692   (family-balanced over 7 families)
robustness_gap     +0.0095   (lower is better)
worst_case         0.9474   (composed_resize0.25+blur0.5+jpeg30)
  flat mean        0.9674   gap +0.0113   (§3.2 literal, secondary)

wrote results/fusion/grid.csv and results/fusion/report.md in 4394.05s


In [43]:
import json

fusion_summary = json.loads(open("results/fusion/summary.json").read())["summary"]
base_summary = json.loads(open("results/baseline/summary.json").read())["summary"]
aug_summary = json.loads(open("results/aug/summary.json").read())["summary"]

# Composite Final Score (README.md "Headline results"): 0.50*AUC_clean + 0.50*AUC_robust,
# AUC_robust = family-balanced mean transformed AUROC.
def final_score(s):
    return 0.5 * s["clean_auroc"] + 0.5 * s["mean_transformed_auroc"]

for name, s in (("baseline", base_summary), ("aug", aug_summary), ("fusion", fusion_summary)):
    print(f"{name:10s}  clean {s['clean_auroc']:.4f}  robust {s['mean_transformed_auroc']:.4f}  "
          f"gap {s['robustness_gap']:.4f}  Final Score {final_score(s):.4f}")

baseline    clean 0.9810  robust 0.9710  gap 0.0099  Final Score 0.9760
aug         clean 0.9779  robust 0.9702  gap 0.0077  Final Score 0.9740
fusion      clean 0.9786  robust 0.9692  gap 0.0095  Final Score 0.9739


### Copy Phase 5 results back to Drive

Same convention as Phase 3/4 -- Colab never pushes; a laptop pulls this from Drive and commits.

In [44]:
shutil.copytree("results/fusion", f"{DRIVE_ROOT}/results/fusion", dirs_exist_ok=True)
print(f"copied to {DRIVE_ROOT}/results/fusion")
print("On the laptop: pull this from Drive into results/fusion/, then git add + commit + push.")

copied to /content/drive/MyDrive/aigc/results/fusion
On the laptop: pull this from Drive into results/fusion/, then git add + commit + push.
